# HierarchicalDet — Kaggle Training Notebook

Trains all three tiers of HierarchicalDet on the DENTEX dataset.

**Session plan (Kaggle T4 = 12h limit):**
```
Session 1: TIER_TO_TRAIN=1, PREV_OUTPUT_DIR=None
Session 2: TIER_TO_TRAIN=2, PREV_OUTPUT_DIR='/kaggle/input/<s1-output>/output'
Session 3: TIER_TO_TRAIN=3, PREV_OUTPUT_DIR='/kaggle/input/<s2-output>/output'
```
After each session: **Save Version → Save & Run All** to commit `/kaggle/working/` as output.

In [ ]:
# ── CONFIGURATION — edit before running ───────────────────────────────────────

GITHUB_REPO     = 'https://github.com/AIscend-Research/HierarchicalDet'
TIER_TO_TRAIN   = 1      # 1, 2, or 3
MAX_ITER        = 40000
NUM_GPUS        = 1

# Previous session output (None for fresh tier-1 start)
# e.g. '/kaggle/input/hierarchicaldet-tier1/output'
PREV_OUTPUT_DIR = None

# Pre-uploaded DENTEX zips as a Kaggle dataset (saves ~30 min per session)
# e.g. '/kaggle/input/dentex-dataset'
KAGGLE_DENTEX_INPUT = None

# HuggingFace token (optional — only needed if KAGGLE_DENTEX_INPUT is None)
HF_TOKEN = None

# ── Derived paths (do not edit) ───────────────────────────────────────────────
import os
WORK_DIR     = '/kaggle/working'
REPO_DIR     = f'{WORK_DIR}/HierarchicalDet'
OFFICIAL_DIR = f'{WORK_DIR}/HierarchicalDet_official'
DATA_ROOT    = f'{WORK_DIR}/sorted/challenge'
OUTPUT_DIR   = f'{WORK_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Tier:            {TIER_TO_TRAIN}')
print(f'Previous output: {PREV_OUTPUT_DIR}')
print(f'DENTEX source:   {KAGGLE_DENTEX_INPUT or "HuggingFace download"}')

In [ ]:
# ── 0. Hardware check ─────────────────────────────────────────────────────────
import torch

print('=== HARDWARE ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -h | grep Mem
!df -h /kaggle/working | tail -1
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU! Set Accelerator → GPU T4 in Settings.'
print(f'GPU:     {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Session timer ──────────────────────────────────────────────────────────
import threading, time as _time

SESSION_START       = _time.time()
SESSION_LIMIT_HOURS = 11.5

def _timer_thread():
    while True:
        remaining = SESSION_LIMIT_HOURS - (_time.time() - SESSION_START) / 3600
        if remaining <= 1.5:
            print(f'\n WARNING: {remaining:.1f}h remaining — save checkpoints soon!')
        if remaining <= 0:
            print('\n Session limit reached.')
            break
        _time.sleep(1800)

threading.Thread(target=_timer_thread, daemon=True).start()
print(f'Session timer started. Limit: {SESSION_LIMIT_HOURS}h')

In [ ]:
# ── 2. Clone repos ────────────────────────────────────────────────────────────
import os, shutil, glob

# Reproduction repo (your fork with custom scripts)
if os.path.exists(REPO_DIR):
    print('Reproduction repo exists — pulling...')
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

# Official HierarchicalDet repo (model code)
if os.path.exists(OFFICIAL_DIR):
    print('Official repo exists — skipping.')
else:
    !git clone https://github.com/ibrahimethemhamamci/HierarchicalDet {OFFICIAL_DIR}
    print('Official repo cloned.')

# Copy all reproduction scripts into official repo
patterns = ['*.py', '*.sh', '*.yaml', '*.md']
copied = []
for pat in patterns:
    for f in glob.glob(f'{REPO_DIR}/{pat}'):
        shutil.copy(f, OFFICIAL_DIR)
        copied.append(os.path.basename(f))
print(f'Copied {len(copied)} scripts to official repo.')

# Verify the critical scripts are present
critical = ['train_net_patched.py', 'dataset_mapper_patched.py', 'organize_dataset.py']
missing = [s for s in critical if not os.path.exists(f'{OFFICIAL_DIR}/{s}')]
if missing:
    raise RuntimeError(f'Critical scripts missing: {missing}\nCheck that GITHUB_REPO is correct and the repo is public.')
print('All critical scripts present.')

%cd {OFFICIAL_DIR}

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
import sys, time, glob, shutil, os, subprocess
t0 = time.time()

!pip install -q fvcore iopath omegaconf timm scipy termcolor yacs tabulate cloudpickle pycocotools huggingface_hub fairscale

# Add official repo to Python path (must come before other imports)
sys.path.insert(0, OFFICIAL_DIR)

# ── Fix 1: bundled pycocotools ────────────────────────────────────────────────
# Copy compiled _mask.so from pip version (bundled lacks it)
import pycocotools as _sys_coco
_sys_dir     = os.path.dirname(_sys_coco.__file__)
_bundled_dir = f'{OFFICIAL_DIR}/pycocotools'
_copied = False
for _f in os.listdir(_sys_dir):
    if '_mask' in _f and _f.endswith('.so'):
        shutil.copy(os.path.join(_sys_dir, _f), _bundled_dir)
        print(f'pycocotools: copied {_f}')
        _copied = True
if not _copied:
    _r = subprocess.run(['find', '/usr/local/lib', '-name', '_mask*.so'],
                        capture_output=True, text=True)
    for _src in _r.stdout.strip().split('\n'):
        if _src and os.path.exists(_src):
            shutil.copy(_src, _bundled_dir)
            print(f'pycocotools: copied {os.path.basename(_src)} (fallback)')

# Fix Python-2-style bare imports in all bundled pycocotools .py files
for _py in glob.glob(f'{_bundled_dir}/*.py'):
    with open(_py) as _f:
        _txt = _f.read()
    _new = _txt.replace('import mask as maskUtils',
                        'from pycocotools import mask as maskUtils')
    if _new != _txt:
        with open(_py, 'w') as _f:
            _f.write(_new)
        print(f'pycocotools: patched {os.path.basename(_py)}')

# Reload so bundled (DENTEX-aware) version is active
for _k in list(sys.modules.keys()):
    if 'pycocotools' in _k:
        del sys.modules[_k]
import pycocotools
print(f'pycocotools from: {os.path.dirname(pycocotools.__file__)}')

# ── Fix 2: bundled detectron2/data/datasets/coco.py ──────────────────────────
# Guard empty category lists — tier1 has categories_2/3 = [] which crashes min()
_coco_d2 = f'{OFFICIAL_DIR}/detectron2/data/datasets/coco.py'
with open(_coco_d2) as _f:
    _txt = _f.read()
_txt = _txt.replace(
    'if not (min(cat_ids_2) == 1 and max(cat_ids_2) == len(cat_ids_2)):',
    'if cat_ids_2 and not (min(cat_ids_2) == 1 and max(cat_ids_2) == len(cat_ids_2)):'
).replace(
    'if not (min(cat_ids_3) == 1 and max(cat_ids_3) == len(cat_ids_3)):',
    'if cat_ids_3 and not (min(cat_ids_3) == 1 and max(cat_ids_3) == len(cat_ids_3)):'
)
with open(_coco_d2, 'w') as _f:
    _f.write(_txt)
print('detectron2/coco.py: patched empty-category guard')

# ── Fix 3: Pillow 10+ ─────────────────────────────────────────────────────────
_replacements = [
    ('Image.LINEAR',    'Image.BILINEAR'),
    ('Image.CUBIC',     'Image.BICUBIC'),
    ('Image.ANTIALIAS', 'Image.LANCZOS'),
]
_patched = 0
for _py in glob.glob(f'{OFFICIAL_DIR}/detectron2/**/*.py', recursive=True):
    with open(_py) as _f:
        _c = _f.read()
    _nc = _c
    for _old, _new in _replacements:
        _nc = _nc.replace(_old, _new)
    if _nc != _c:
        with open(_py, 'w') as _f:
            _f.write(_nc)
        _patched += 1
print(f'Pillow 10 patch: {_patched} files updated.')

import detectron2
print(f'detectron2: {detectron2.__version__}')
print(f'Install time: {time.time()-t0:.0f}s')

In [ ]:
# ── 4. Download Swin-B backbone ───────────────────────────────────────────────
import urllib.request, pickle, torch, time, os

os.makedirs(f'{OFFICIAL_DIR}/models', exist_ok=True)
pkl_path = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'

if not os.path.exists(pkl_path):
    url = ('https://github.com/SwinTransformer/storage/releases/download/'
           'v1.0.0/swin_base_patch4_window7_224_22k.pth')
    pth_path = pkl_path.replace('.pkl', '.pth')
    print('Downloading Swin-B backbone (~340 MB)...')
    t0 = time.time()
    urllib.request.urlretrieve(url, pth_path)
    print(f'Downloaded in {time.time()-t0:.0f}s')
    ckpt = torch.load(pth_path, map_location='cpu')
    d2_ckpt = {'model': ckpt.get('model', ckpt),
               '__author__': 'third_party', 'matching_heuristics': True}
    with open(pkl_path, 'wb') as f:
        pickle.dump(d2_ckpt, f)
    print(f'Backbone saved: {pkl_path}')
else:
    print('Backbone already present.')

In [ ]:
# ── 5. Set up DENTEX dataset ──────────────────────────────────────────────────
import os, time

DENTEX_RAW = f'{WORK_DIR}/sorted/dentex_raw/DENTEX'

def _symlink_sorted():
    dst = f'{OFFICIAL_DIR}/sorted'
    src = f'{WORK_DIR}/sorted'
    if not os.path.exists(dst) and not os.path.islink(dst):
        os.symlink(src, dst)
        print(f'Linked {src} → {dst}')

if os.path.exists(f'{DATA_ROOT}/train_merged_disease_coco3class_onlyd_fixed.json'):
    print('Dataset already organized. Skipping.')

elif KAGGLE_DENTEX_INPUT:
    # ── Fast path: pre-uploaded Kaggle dataset ────────────────────────────────
    import shutil, zipfile
    print(f'Using mounted dataset: {KAGGLE_DENTEX_INPUT}')
    os.makedirs(DENTEX_RAW, exist_ok=True)
    val_json = f'{KAGGLE_DENTEX_INPUT}/validation_triple.json'
    if os.path.exists(val_json):
        shutil.copy(val_json, f'{DENTEX_RAW}/validation_triple.json')
    for zname in ['training_data.zip', 'validation_data.zip', 'test_data.zip']:
        zpath = f'{KAGGLE_DENTEX_INPUT}/{zname}'
        if os.path.exists(zpath):
            print(f'Extracting {zname}...')
            t0 = time.time()
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(f'{WORK_DIR}/sorted')
            print(f'  Done in {time.time()-t0:.0f}s')
        else:
            print(f'  WARNING: {zname} not found in {KAGGLE_DENTEX_INPUT}')
    _symlink_sorted()
    !python {OFFICIAL_DIR}/organize_dataset.py

else:
    # ── Slow path: stream-extract from HuggingFace via remotezip ─────────────
    # remotezip fetches individual files via HTTP range requests — the zip is
    # never stored on disk, so we only use ~11 GB (extracted) not ~22 GB.
    !pip install -q remotezip
    from remotezip import RemoteZip
    from huggingface_hub import hf_hub_url
    import shutil

    os.makedirs(DENTEX_RAW, exist_ok=True)
    os.makedirs(f'{WORK_DIR}/sorted', exist_ok=True)

    HF_REPO = 'ibrahimhamamci/DENTEX'

    zips = [
        ('DENTEX/training_data.zip',   f'{WORK_DIR}/sorted'),
        ('DENTEX/validation_data.zip', f'{WORK_DIR}/sorted'),
        ('DENTEX/test_data.zip',       f'{WORK_DIR}/sorted'),
    ]

    for hf_path, extract_to in zips:
        zname = os.path.basename(hf_path)
        url = hf_hub_url(repo_id=HF_REPO, filename=hf_path, repo_type='dataset')
        print(f'Streaming {zname}...')
        t0 = time.time()
        with RemoteZip(url) as rz:
            rz.extractall(extract_to)
        print(f'  Done in {(time.time()-t0)/60:.1f} min')
        !df -h /kaggle/working | tail -1

    # Download validation_triple.json directly (small standalone file)
    val_json_url = hf_hub_url(repo_id=HF_REPO,
                               filename='DENTEX/validation_quadrant_enumeration_disease.json',
                               repo_type='dataset')
    import urllib.request
    try:
        urllib.request.urlretrieve(val_json_url,
                                   f'{DENTEX_RAW}/validation_triple.json')
        print('Downloaded validation JSON.')
    except Exception as e:
        # Try alternate filename
        for fname in ['validation_triple.json', 'validation_quadrant_enumeration_disease.json']:
            try:
                alt_url = hf_hub_url(repo_id=HF_REPO, filename=f'DENTEX/{fname}', repo_type='dataset')
                urllib.request.urlretrieve(alt_url, f'{DENTEX_RAW}/validation_triple.json')
                print(f'Downloaded validation JSON ({fname}).')
                break
            except Exception:
                continue

    _symlink_sorted()
    !python {OFFICIAL_DIR}/organize_dataset.py

print('\nDataset ready:')
!ls {DATA_ROOT}

In [ ]:
# ── 6. Restore checkpoints from previous session ──────────────────────────────
import shutil, os

os.makedirs(f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}', exist_ok=True)

if PREV_OUTPUT_DIR:
    print(f'Restoring from: {PREV_OUTPUT_DIR}')
    for item in os.listdir(PREV_OUTPUT_DIR):
        src = os.path.join(PREV_OUTPUT_DIR, item)
        dst = os.path.join(f'{OFFICIAL_DIR}/output', item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f'  {item}')
    print('Restore complete.')
else:
    print('No previous output (fresh start).')

!find {OFFICIAL_DIR}/output -name '*.pth' 2>/dev/null | head -10 || echo 'No checkpoints yet.'

In [ ]:
# ── 7. Environment + config paths ─────────────────────────────────────────────
import os

os.environ['DATA_ROOT']       = DATA_ROOT
os.environ['DENTEX_TIER']     = f'tier{TIER_TO_TRAIN}'
os.environ['USE_NOISY_BOXES'] = '0'

CONFIG  = f'{OFFICIAL_DIR}/configs/diffdet.custom.swinbase.nonpretrain.yaml'

if TIER_TO_TRAIN == 1:
    WEIGHTS = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'
elif TIER_TO_TRAIN == 2:
    WEIGHTS  = f'{OFFICIAL_DIR}/output/tier1/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier1_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier1_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'
elif TIER_TO_TRAIN == 3:
    WEIGHTS  = f'{OFFICIAL_DIR}/output/tier2/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier2_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier2_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'

assert os.path.exists(WEIGHTS), f'Weights not found: {WEIGHTS}'
assert os.path.exists(CONFIG),  f'Config not found: {CONFIG}'

print(f'Tier:            {TIER_TO_TRAIN}')
print(f'Weights:         {WEIGHTS}')
print(f'Config:          {CONFIG}')
print(f'USE_NOISY_BOXES: {os.environ["USE_NOISY_BOXES"]}')
print(f'DATA_ROOT:       {DATA_ROOT}')

In [ ]:
# ── 8. Generate noisy boxes (tiers 2 and 3 only) ─────────────────────────────
if TIER_TO_TRAIN > 1 and os.environ.get('USE_NOISY_BOXES') != '1':
    prev = TIER_TO_TRAIN - 1
    prev_weights = f'{OFFICIAL_DIR}/output/tier{prev}/model_final.pth'
    assert os.path.exists(prev_weights), f'Tier {prev} checkpoint not found: {prev_weights}'
    os.makedirs(f'{OFFICIAL_DIR}/noisy_boxes', exist_ok=True)
    for split, ann_json, img_dir, out_file in [
        ('train',
         f'{DATA_ROOT}/train_merged_disease_coco3class_onlyd_fixed.json',
         f'{DATA_ROOT}/for_coco_disease_train',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_train_boxes.json'),
        ('val',
         f'{DATA_ROOT}/test_merged_disease_coco3class.json',
         f'{DATA_ROOT}/for_coco_disease_test',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_val_boxes.json'),
    ]:
        print(f'Generating noisy boxes ({split})...')
        !python {OFFICIAL_DIR}/phase2_generate_noisy_boxes.py \
            --config-file {CONFIG} \
            --weights {prev_weights} \
            --tier {prev - 1} \
            --ann-json {ann_json} \
            --img-dir {img_dir} \
            --out {out_file}
    os.environ['NOISY_BOX_TRAIN'] = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_train_boxes.json'
    os.environ['NOISY_BOX_VAL']   = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev}_val_boxes.json'
    os.environ['USE_NOISY_BOXES'] = '1'
    print('Noisy boxes generated.')
else:
    print(f'Tier {TIER_TO_TRAIN}: noisy box step skipped.')

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
import os, re, subprocess
from IPython.display import clear_output

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

out_dir    = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}'
final_ckpt = f'{out_dir}/model_final.pth'
log_path   = f'{out_dir}/train.log'
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(final_ckpt):
    print(f'model_final.pth already exists — tier {TIER_TO_TRAIN} done.')
else:
    cmd = [
        'python', '-u', f'{OFFICIAL_DIR}/train_net_patched.py',
        '--config-file', CONFIG,
        '--num-gpus', str(NUM_GPUS),
        'MODEL.WEIGHTS',             WEIGHTS,
        'OUTPUT_DIR',                out_dir,
        'SOLVER.MAX_ITER',           str(MAX_ITER),
        'SOLVER.IMS_PER_BATCH',      '1',
        'MODEL.SWIN.USE_CHECKPOINT', 'True',
        'DATALOADER.NUM_WORKERS',    '0',
        'TEST.EVAL_PERIOD',          '0',
        'SEED',                      '40244023',
    ]

    stats  = dict(iter=0, loss='…', lr='…', eta='…')
    errors = []

    def _render(done=False):
        pct    = stats['iter'] / MAX_ITER * 100
        filled = int(pct / 2)
        bar    = '█' * filled + '░' * (50 - filled)
        clear_output(wait=True)
        status = '✓ DONE' if done else 'running…'
        print(f'Tier {TIER_TO_TRAIN} training — {status}')
        print(f'[{bar}] {pct:.1f}%')
        print(f'  iter {stats["iter"]:>6} / {MAX_ITER}')
        print(f'  loss {stats["loss"]}   lr {stats["lr"]}   ETA {stats["eta"]}')
        print(f'  log → {log_path}')
        for e in errors[-3:]:
            print(f'  !! {e}')

    _render()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=os.environ.copy())

    with open(log_path, 'w') as lf:
        for line in proc.stdout:
            lf.write(line); lf.flush()
            m = re.search(
                r'iter:\s*(\d+).*?total_loss:\s*([\d.]+).*?lr:\s*([\d.e+\-]+).*?eta:\s*(\S+)',
                line)
            if m:
                stats.update(iter=int(m.group(1)), loss=m.group(2),
                             lr=m.group(3), eta=m.group(4))
                _render()
            if any(k in line for k in ('Error', 'ERROR', 'Traceback', 'CUDA out')):
                errors.append(line.rstrip())
                _render()

    proc.wait()
    if os.path.exists(final_ckpt):
        _render(done=True)
        print(f'\nSaved: {final_ckpt}  ({os.path.getsize(final_ckpt)/1e6:.0f} MB)')
    else:
        _render()
        print(f'\nStopped early — partial checkpoints in {out_dir}')
        print('Resume next session with PREV_OUTPUT_DIR pointing here.')

In [ ]:
# ── 10. Evaluate ──────────────────────────────────────────────────────────────
final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if not os.path.exists(final_ckpt):
    print('No final checkpoint — run training cell first.')
else:
    eval_out = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}_eval'
    os.makedirs(eval_out, exist_ok=True)
    print(f'Evaluating tier {TIER_TO_TRAIN}...')
    !python {OFFICIAL_DIR}/train_net_patched.py \
        --config-file {CONFIG} \
        --eval-only \
        MODEL.WEIGHTS {final_ckpt} \
        OUTPUT_DIR {eval_out}
    !python {OFFICIAL_DIR}/phase2_collect_results.py \
        --log-files {eval_out}/log.txt \
        --tier-names "Tier{TIER_TO_TRAIN}"

In [ ]:
# ── 11. Save outputs for next session ─────────────────────────────────────────
# Copies checkpoints into /kaggle/working/ which Kaggle saves as notebook output.
# After this cell: click Save Version → Save & Run All (select 'Only files').
import os, shutil

for src, dst in [
    (f'{OFFICIAL_DIR}/output',      f'{WORK_DIR}/output'),
    (f'{OFFICIAL_DIR}/noisy_boxes', f'{WORK_DIR}/noisy_boxes'),
]:
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {dst}')

print('\n=== Saved files ===')
for root, _, files in os.walk(f'{WORK_DIR}/output'):
    for f in files:
        path = os.path.join(root, f)
        print(f'  {path.replace(WORK_DIR,"")}  ({os.path.getsize(path)/1e6:.0f} MB)')

next_tier = TIER_TO_TRAIN + 1
if next_tier <= 3:
    print(f'\n=== Next session ===')
    print(f'1. Save Version → Save & Run All → Only files')
    print(f'2. New session → Add Data → this notebook output')
    print(f'3. Set TIER_TO_TRAIN = {next_tier}')
    print(f'4. Set PREV_OUTPUT_DIR = "/kaggle/input/<dataset-name>/output"')
    print(f'5. Run All')
else:
    print('\nAll 3 tiers complete! Proceed to evaluation and extensions.')

In [ ]:
# ── 12. (Optional) Runtime benchmark ─────────────────────────────────────────
final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if not os.path.exists(final_ckpt):
    print('No checkpoint — skipping benchmark.')
else:
    for device, n in [('cuda', 20), ('cpu', 5)]:
        out = f'{WORK_DIR}/output/runtime_tier{TIER_TO_TRAIN}_{device}.json'
        !python {OFFICIAL_DIR}/phase2_runtime_benchmark.py \
            --config-file {CONFIG} \
            --weights {final_ckpt} \
            --tier {TIER_TO_TRAIN - 1} \
            --n-images {n} \
            --device {device} \
            --out {out}